## Transforming data BI ready and moving to gold layer ##

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/gold/fact_order_returns"

In [0]:
df_silver_returns =spark.readStream.format("delta").option("readChangeData","true").table(f"{catalog_name}.silver.slv_order_returns")

In [0]:
df_gold_returns =df_silver_returns.withColumn("dateid",F.date_format(F.col("order_dt"),"yyyyMMdd"))

In [0]:
df_gold_returns = df_gold_returns.withColumn("return_days",F.date_diff(F.col("return_ts"),F.col("order_dt")) )

In [0]:
df_gold_returns = df_gold_returns.withColumn("within_policy",F.when(F.col("return_days") <= 15,1).otherwise(0))
df_gold_returns = df_gold_returns.withColumn("is_late_return",F.when(F.col("return_days") > 15 ,1).otherwise(0))

In [0]:
def upsert_to_gold(microBatchDF, batchId):
    ##microBatchDF = microBatchDF.drop(
    ##    "_change_type",
      ##  "_commit_version",
        ##"_commit_timestamp"
   ## )

    table_name = f"{catalog_name}.gold.gld_fact_order_returns"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.order_id = batch_table.order_id AND gold_table.order_dt = batch_table.order_dt AND gold_table.return_ts = batch_table.return_ts",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:


df_gold_returns.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).start().awaitTermination()